# Cross-Validation Setup

After running `ag-data-cleaning.ipynb` and `ag-data-selection.ipynb` (under `data-processing/`), use this notebook to split `test.xyz` (the training set of your DFT relaxations) into 4 CV folds.

### K-fold Cross Validation

After splitting off a test set for later, K-fold CV divides training data into _K_ equal-sized subsets ("folds"). The model is trained on _K-1_ folds and tested on the remaining one. This repeated _K_ times.

The primary goal of cross-validation is to ensure your model generalizes well to unseen data, mitigating sampling bias and reducing the variance associated with a single, arbitrary train-test split.

For example, this notebook splits the test set into 4 folds: `A | B | C | D`
```
Fold 0:  train = B+C+D   valid = A
Fold 1:  train = A+C+D   valid = B
Fold 2:  train = A+B+D   valid = C
Fold 3:  train = A+B+C   valid = D
```

In [1]:
from pathlib import Path

from ase.io import read, write
from sklearn.model_selection import KFold, train_test_split

In [2]:
INPUT = Path("../../training-data/train.xyz")
OUTDIR = Path("../../training-data/folds")
TEST_FRACTION = 0.1
N_FOLDS = 4
SEED = 123  # matches existing _valid_indices_123.txt convention

OUTDIR.mkdir(parents=True, exist_ok=True)

In [3]:
configs = read(INPUT, index=":", format="extxyz")
n_total = len(configs)
print(f"Loaded {n_total} configs from {INPUT}")

Loaded 356 configs from ../../training-data/train.xyz


## 0. Split train/test sets

In [6]:
train_pool, test_set = train_test_split(
    configs, test_size=TEST_FRACTION, random_state=SEED
)

test_path = OUTDIR / "test.xyz"
write(test_path, test_set, format="extxyz")
print(f"Wrote {len(test_set)} configs -> {test_path} (held out, untouched)")

Wrote 36 configs -> ../../training-data/folds/test.xyz (held out, untouched)


## 1. K-fold split remaining training data

In [7]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold_idx, (train_idx, valid_idx) in enumerate(kf.split(train_pool)):
    fold_train = [train_pool[i] for i in train_idx]
    fold_valid = [train_pool[i] for i in valid_idx]

    train_path = OUTDIR / f"fold{fold_idx}_train.xyz"
    valid_path = OUTDIR / f"fold{fold_idx}_valid.xyz"

    write(train_path, fold_train, format="extxyz")
    write(valid_path, fold_valid, format="extxyz")

    print(f"Fold {fold_idx}: train={len(fold_train)} -> {train_path}, "
          f"valid={len(fold_valid)} -> {valid_path}")

Fold 0: train=240 -> ../../training-data/folds/fold0_train.xyz, valid=80 -> ../../training-data/folds/fold0_valid.xyz
Fold 1: train=240 -> ../../training-data/folds/fold1_train.xyz, valid=80 -> ../../training-data/folds/fold1_valid.xyz
Fold 2: train=240 -> ../../training-data/folds/fold2_train.xyz, valid=80 -> ../../training-data/folds/fold2_valid.xyz
Fold 3: train=240 -> ../../training-data/folds/fold3_train.xyz, valid=80 -> ../../training-data/folds/fold3_valid.xyz
